# Proceso de Etiquetado de Complejidades

**Objetivo:** Documentar y ejecutar el proceso de etiquetado automático de algoritmos con sus complejidades
**Duración estimada:** 35 minutos

---

## Contenido

1. [Setup](#setup)
2. [Estrategias de Etiquetado](#estrategias-de-etiquetado)
3. [Etiquetado Automático con el Motor de Análisis](#etiquetado-automatico)
4. [Etiquetado Manual como Referencia](#etiquetado-manual)
5. [Resolución de Conflictos entre Etiquetas](#resolucion-conflictos)
6. [Etiquetado de Patrones Algorítmicos](#etiquetado-patrones)
7. [Métricas de Calidad del Etiquetado](#metricas-calidad)
8. [Pipeline Completo de Etiquetado](#pipeline-completo)

---

## 1. Setup

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter, defaultdict
from enum import Enum

sys.path.insert(0, '../..')

from dataset_generator import (
    ComplexityLabeler,
    ComplexityLabel,
    label_algorithm
)
from app.core.parser import parse_pseudocode
from app.core.analyzer import AnalyzerEngine
from app.core.patterns import PatternDetector, PatternType

import matplotlib.pyplot as plt
import numpy as np

print("Setup completado")

---

## 2. Estrategias de Etiquetado

El etiquetado es el proceso de asignar metadatos descriptivos a cada algoritmo en el dataset.
El sistema implementa tres estrategias complementarias:

| Estrategia | Fuente | Confianza | Uso |
|------------|--------|-----------|-----|
| Automático (motor) | AnalyzerEngine | Variable | Escala masivamente |
| Manual (experto) | Ground truth | Alta | Validación y corrección |
| Consenso (híbrido) | Ambas fuentes | Alta | Dataset final |

### Estructura de una Etiqueta Completa

In [ ]:
class EtiquetaAlgoritmo:
    """Representa una etiqueta completa de un algoritmo en el dataset."""
    
    def __init__(self, algoritmo_id, codigo):
        self.id = algoritmo_id
        self.codigo = codigo
        
        # Complejidad temporal
        self.big_o = None           # Peor caso: O(n), O(n^2), etc.
        self.omega = None           # Mejor caso
        self.theta = None           # Caso promedio
        
        # Complejidad espacial
        self.espacio = None         # S(n) = O(1), O(n), etc.
        
        # Patrón algorítmico
        self.patron = None          # SORTING, SEARCHING, RECURSIVE, etc.
        self.confianza_patron = 0.0
        
        # Estructuras de datos
        self.estructuras = []       # ["ARRAY", "STACK", etc.]
        
        # Metadatos
        self.fuente_etiqueta = None # "automatico", "manual", "consenso"
        self.confianza_global = 0.0
        self.notas = ""
    
    def to_dict(self):
        return {k: str(v) if v is not None else None 
                for k, v in self.__dict__.items() 
                if not k.startswith('_')}

print("Estructura de etiqueta definida")

---

## 3. Etiquetado Automático con el Motor de Análisis

In [ ]:
engine = AnalyzerEngine()
pattern_detector = PatternDetector()

def etiquetar_automaticamente(codigo, algoritmo_id="auto"):
    """
    Etiqueta automáticamente un algoritmo usando el motor de análisis.
    
    Retorna un diccionario con las etiquetas inferidas y su nivel de confianza.
    """
    etiqueta = EtiquetaAlgoritmo(algoritmo_id, codigo)
    etiqueta.fuente_etiqueta = "automatico"
    
    errores = []
    
    # Paso 1: Parsing
    try:
        ast = parse_pseudocode(codigo)
    except Exception as e:
        return None, f"Error de parsing: {str(e)[:60]}"
    
    # Paso 2: Análisis de complejidad
    try:
        analysis = engine.analyze(ast)
        etiqueta.big_o = (getattr(analysis, 'big_o', None) or
                          getattr(analysis, 'time_complexity', {}).get('big_o'))
        etiqueta.omega = (getattr(analysis, 'omega', None) or
                          getattr(analysis, 'time_complexity', {}).get('omega'))
        espacio = (getattr(analysis, 'space_complexity', None) or
                   getattr(analysis, 'space', None))
        etiqueta.espacio = str(espacio) if espacio else "O(1)"
        
        # Confianza basada en si se obtuvo un resultado
        if etiqueta.big_o:
            etiqueta.confianza_global += 0.4
    except Exception as e:
        errores.append(f"analisis: {str(e)[:40]}")
    
    # Paso 3: Detección de patrones
    try:
        detection = pattern_detector.detect(ast)
        etiqueta.patron = str(getattr(detection, 'primary_pattern', None))
        etiqueta.confianza_patron = float(getattr(detection, 'primary_confidence', 0.0))
        if etiqueta.confianza_patron > 0.3:
            etiqueta.confianza_global += 0.3 * etiqueta.confianza_patron
    except Exception as e:
        errores.append(f"patron: {str(e)[:40]}")
    
    if errores:
        etiqueta.notas = "; ".join(errores)
    
    return etiqueta, None


# Algoritmos de prueba para el proceso de etiquetado
ALGORITMOS_A_ETIQUETAR = [
    ("algo_001", """
algorithm insertionSort(A[], n)
begin
    for i <- 2 to n do
        key <- A[i]
        j <- i - 1
        while (j > 0 and A[j] > key) do
            A[j + 1] <- A[j]
            j <- j - 1
        end
        A[j + 1] <- key
    end
end
"""),
    ("algo_002", """
algorithm contarPares(A[], n)
begin
    count <- 0
    for i <- 1 to n do
        if (A[i] mod 2 = 0) then
            count <- count + 1
        end
    end
    return count
end
"""),
    ("algo_003", """
algorithm potencia(base, exp)
begin
    if (exp = 0) then
        return 1
    end
    return base * potencia(base, exp - 1)
end
"""),
    ("algo_004", """
algorithm quickSort(A[], low, high)
begin
    if (low < high) then
        pivot <- A[high]
        i <- low - 1
        for j <- low to high - 1 do
            if (A[j] <= pivot) then
                i <- i + 1
                temp <- A[i]
                A[i] <- A[j]
                A[j] <- temp
            end
        end
        temp <- A[i + 1]
        A[i + 1] <- A[high]
        A[high] <- temp
        call quickSort(A, low, i)
        call quickSort(A, i + 2, high)
    end
end
"""),
]

print("PROCESO DE ETIQUETADO AUTOMÁTICO:")
etiquetas_generadas = []
for alg_id, codigo in ALGORITMOS_A_ETIQUETAR:
    etiqueta, error = etiquetar_automaticamente(codigo, alg_id)
    if etiqueta:
        etiquetas_generadas.append(etiqueta)
        print(f"\n[{alg_id}]")
        print(f"  BigO:    {etiqueta.big_o}")
        print(f"  Omega:   {etiqueta.omega}")
        print(f"  Espacio: {etiqueta.espacio}")
        print(f"  Patrón:  {etiqueta.patron} (conf={etiqueta.confianza_patron:.2f})")
        print(f"  Conf. global: {etiqueta.confianza_global:.2f}")
    else:
        print(f"[{alg_id}] ERROR: {error}")

---

## 4. Etiquetado Manual como Referencia

In [ ]:
# Ground truth definido manualmente por experto
ETIQUETAS_MANUALES = {
    "algo_001": {
        "big_o": "O(n^2)",
        "omega": "O(n)",
        "espacio": "O(1)",
        "patron": "SORTING",
        "estructuras": ["ARRAY"],
        "nombre": "Insertion Sort"
    },
    "algo_002": {
        "big_o": "O(n)",
        "omega": "O(n)",
        "espacio": "O(1)",
        "patron": "BRUTE_FORCE",
        "estructuras": ["ARRAY"],
        "nombre": "Conteo de pares"
    },
    "algo_003": {
        "big_o": "O(n)",
        "omega": "O(1)",
        "espacio": "O(n)",
        "patron": "RECURSIVE",
        "estructuras": [],
        "nombre": "Potencia recursiva"
    },
    "algo_004": {
        "big_o": "O(n^2)",
        "omega": "O(n log n)",
        "espacio": "O(log n)",
        "patron": "DIVIDE_AND_CONQUER",
        "estructuras": ["ARRAY"],
        "nombre": "Quick Sort"
    },
}

print("ETIQUETAS MANUALES (Ground Truth):")
for alg_id, etiqueta in ETIQUETAS_MANUALES.items():
    print(f"  [{alg_id}] {etiqueta['nombre']}: BigO={etiqueta['big_o']}, Patron={etiqueta['patron']}")

---

## 5. Resolución de Conflictos entre Etiquetas

In [ ]:
def resolver_conflicto(auto, manual):
    """
    Resuelve conflictos entre etiqueta automática y manual.
    
    Estrategia:
    - Si coinciden: alta confianza
    - Si difieren: usar etiqueta manual (más confiable) y registrar discrepancia
    """
    resultado = manual.copy()
    discrepancias = []
    
    if auto is None:
        resultado["fuente"] = "manual"
        resultado["confianza"] = 0.9
        return resultado, discrepancias
    
    # Comparar Big O
    if str(auto.big_o) != manual.get("big_o"):
        discrepancias.append(
            f"BigO: auto={auto.big_o}, manual={manual.get('big_o')}"
        )
    
    # Comparar patrón
    patron_auto = str(auto.patron).replace("PatternType.", "")
    patron_manual = manual.get("patron", "")
    if patron_auto != patron_manual:
        discrepancias.append(
            f"Patron: auto={patron_auto}, manual={patron_manual}"
        )
    
    coincide = len(discrepancias) == 0
    resultado["fuente"] = "consenso" if coincide else "manual_corregido"
    resultado["confianza"] = 0.95 if coincide else 0.8
    resultado["discrepancias_con_auto"] = discrepancias
    
    return resultado, discrepancias

print("RESOLUCIÓN DE CONFLICTOS:")
etiquetas_finales = {}
total_discrepancias = 0
for etiqueta_auto in etiquetas_generadas:
    alg_id = etiqueta_auto.id
    manual = ETIQUETAS_MANUALES.get(alg_id)
    
    if manual:
        final, discrepancias = resolver_conflicto(etiqueta_auto, manual)
        etiquetas_finales[alg_id] = final
        total_discrepancias += len(discrepancias)
        estado = "COINCIDE" if not discrepancias else f"DISCREPANCIA ({len(discrepancias)})"
        print(f"  [{alg_id}]: {estado}")
        for d in discrepancias:
            print(f"    -> {d}")

print(f"\nTotal discrepancias encontradas: {total_discrepancias}")

---

## 6. Etiquetado de Patrones Algorítmicos

In [ ]:
# Estadísticas de patrones en el dataset etiquetado
patron_counter = Counter()
big_o_counter = Counter()

for alg_id, etiqueta in etiquetas_finales.items():
    patron_counter[etiqueta.get("patron", "DESCONOCIDO")] += 1
    big_o_counter[etiqueta.get("big_o", "?")] += 1

print("DISTRIBUCIÓN DE PATRONES EN DATASET ETIQUETADO:")
for patron, cantidad in patron_counter.most_common():
    barra = "#" * (cantidad * 5)
    print(f"  {patron:<25}: {barra} ({cantidad})")

print("\nDISTRIBUCIÓN DE COMPLEJIDADES:")
for big_o, cantidad in big_o_counter.most_common():
    barra = "#" * (cantidad * 5)
    print(f"  {str(big_o):<15}: {barra} ({cantidad})")

---

## 7. Métricas de Calidad del Etiquetado

In [ ]:
def calcular_metricas_etiquetado(etiquetas_auto, ground_truth):
    """Calcula métricas de calidad del proceso de etiquetado automático."""
    
    total = len(ground_truth)
    correctos_big_o = 0
    correctos_patron = 0
    suma_confianza = 0
    
    for alg_id, verdad in ground_truth.items():
        auto = next((e for e in etiquetas_auto if e.id == alg_id), None)
        if auto is None:
            continue
        
        if str(auto.big_o) == verdad.get("big_o"):
            correctos_big_o += 1
        
        patron_auto = str(auto.patron).replace("PatternType.", "")
        if patron_auto == verdad.get("patron"):
            correctos_patron += 1
        
        suma_confianza += auto.confianza_global
    
    return {
        "precision_big_o": correctos_big_o / total if total > 0 else 0,
        "precision_patron": correctos_patron / total if total > 0 else 0,
        "confianza_promedio": suma_confianza / total if total > 0 else 0,
        "total_evaluados": total
    }

metricas = calcular_metricas_etiquetado(etiquetas_generadas, ETIQUETAS_MANUALES)
print("MÉTRICAS DE CALIDAD DEL ETIQUETADO AUTOMÁTICO:")
print(f"Precisión Big O:          {metricas['precision_big_o']:.1%}")
print(f"Precisión Patrón:         {metricas['precision_patron']:.1%}")
print(f"Confianza promedio:       {metricas['confianza_promedio']:.2f}")
print(f"Algoritmos evaluados:     {metricas['total_evaluados']}")

---

## 8. Pipeline Completo de Etiquetado

In [ ]:
def pipeline_etiquetado(lista_algoritmos, ground_truth=None):
    """
    Pipeline completo de etiquetado para un conjunto de algoritmos.
    
    Pasos:
    1. Etiquetado automático
    2. Resolución de conflictos (si hay ground truth)
    3. Exportación del dataset etiquetado
    """
    print("INICIANDO PIPELINE DE ETIQUETADO")
    
    resultados = []
    errores = []
    
    for alg_id, codigo in lista_algoritmos:
        etiqueta_auto, error = etiquetar_automaticamente(codigo, alg_id)
        
        if error:
            errores.append({"id": alg_id, "error": error})
            continue
        
        if ground_truth and alg_id in ground_truth:
            etiqueta_final, _ = resolver_conflicto(etiqueta_auto, ground_truth[alg_id])
        else:
            etiqueta_final = etiqueta_auto.to_dict()
            etiqueta_final["fuente"] = "automatico"
        
        etiqueta_final["id"] = alg_id
        etiqueta_final["codigo"] = codigo
        resultados.append(etiqueta_final)
    
    print(f"Etiquetados exitosamente: {len(resultados)}/{len(lista_algoritmos)}")
    print(f"Errores encontrados:       {len(errores)}")
    
    return resultados, errores

resultados_pipeline, errores_pipeline = pipeline_etiquetado(
    ALGORITMOS_A_ETIQUETAR,
    ground_truth=ETIQUETAS_MANUALES
)

# Guardar resultado
output_file = Path("../../data/datasets/labeled_algorithms.json")
output_file.parent.mkdir(parents=True, exist_ok=True)
with open(output_file, "w", encoding="utf-8") as f:
    json.dump({"metadata": metricas, "ejemplos": resultados_pipeline}, f, indent=2, ensure_ascii=False)

print(f"\nDataset etiquetado guardado en: {output_file}")

---

## Proximos Pasos

- **dataset_validation.ipynb**: Validar la calidad y distribución del dataset completo
- **synthetic_algorithms.ipynb**: Generar más variaciones sintéticas para balancear el dataset